# LeetCode 3367 — Maximize Sum of Weights after Edge Removals bằng Python

- Link: https://leetcode.com/problems/maximize-sum-of-weights-after-edge-removals/
- Độ khó: **Hard**
- Chủ đề: **DP trên cây** với hai trạng thái (cạnh lên cha **giữ** hay **bỏ**), chọn **k "lợi ích" lớn nhất** trong các con, duyệt cây **không đệ quy**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximizeSumOfWeights(self, edges: List[List[int]], k: int) -> int:
```

## 1. Mục tiêu học

1. Đặt bài: giữ một tập cạnh của cây sao cho **bậc mỗi đỉnh ≤ k**, tổng trọng số lớn nhất.
2. DP trên cây: `f[u][0]` = tốt nhất trong cây con `u` khi `u` còn được dùng **k** cạnh con; `f[u][1]` = khi cạnh `u – cha` đã giữ nên `u` chỉ còn **k − 1** cạnh con.
3. Với mỗi con `v`: giá trị cơ sở `f[v][0]` (bỏ cạnh `u–v`), lợi ích nếu giữ cạnh `gain = w + f[v][1] − f[v][0]`; chọn tối đa `k` (hoặc `k − 1`) lợi ích **dương** lớn nhất.
4. Duyệt cây theo thứ tự BFS ngược để tránh đệ quy sâu.
5. Thấy vì sao tham lam "giữ cạnh nặng trước" sai.

## 2. Đề bài & ví dụ

Cho cây `n` đỉnh, cạnh có trọng số. Xóa một số cạnh sao cho mỗi đỉnh nối với **tối đa `k`** đỉnh khác; tối đa hóa tổng trọng số cạnh còn lại.

**Ví dụ 1:** `edges = [[0,1,4],[0,2,2],[2,3,12],[2,4,6]], k = 2` → `22` (bỏ cạnh `0–2`).

**Ví dụ 2:** `k = 3`, không cần bỏ cạnh nào → `65`.

**Ràng buộc:** `n ≤ 10^5`, `1 ≤ k ≤ n − 1`, `w ≤ 10^6`.

In [1]:
from typing import List
import itertools

examples = [
    ([[0, 1, 4], [0, 2, 2], [2, 3, 12], [2, 4, 6]], 2, 22),
    ([[0, 1, 5], [1, 2, 10], [0, 3, 15], [3, 4, 20], [3, 5, 5], [0, 6, 10]], 3, 65),
]
for e, k, exp in examples:
    print(len(e) + 1, 'đỉnh, k =', k, '-> kỳ vọng', exp)

5 đỉnh, k = 2 -> kỳ vọng 22
7 đỉnh, k = 3 -> kỳ vọng 65


## 3. Kiến thức nền

### 3.1. Duyệt cây không đệ quy (thứ tự BFS, xử lý ngược)

Xử lý đỉnh theo thứ tự BFS **ngược** bảo đảm mọi con được xử lý trước cha.

In [2]:
def bfs_order(n, edges):
    g = [[] for _ in range(n)]
    for u, v, w in edges:
        g[u].append((v, w)); g[v].append((u, w))
    parent, pw, order = [-1] * n, [0] * n, [0]
    seen = [False] * n; seen[0] = True
    for u in order:
        for v, w in g[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; pw[v] = w; order.append(v)
    return parent, pw, order

print(bfs_order(5, examples[0][0]))

([-1, 0, 0, 2, 2], [0, 4, 2, 12, 6], [0, 1, 2, 3, 4])


### 3.2. Chọn tối đa `t` phần tử dương lớn nhất

Sắp giảm dần, lấy `t` phần tử đầu, bỏ phần tử `≤ 0`. Hoặc `heapq.nlargest(t, …)`.

In [3]:
def top_positive(vals, t):
    return sum(x for x in sorted(vals, reverse=True)[:max(t, 0)] if x > 0)

print(top_positive([5, -1, 3, 8, 0], 2), top_positive([5, -1, 3, 8, 0], 0), top_positive([-2, -3], 2))

13 0 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1, gốc 0, `k = 2`. Đỉnh 2 có ba con… (cha 0, con 3, 4).

- Lá 1, 3, 4: `f = (0, 0)`.
- Đỉnh 2 (con 3: lợi ích 12; con 4: lợi ích 6): `f[2][0]` = top 2 = `18`; `f[2][1]` = top 1 = `12`.
- Đỉnh 0 (con 1: lợi ích `4 + 0 − 0 = 4`; con 2: lợi ích `2 + f[2][1] − f[2][0] = 2 + 12 − 18 = −4`, cơ sở `f[2][0] = 18`): `f[0][0] = 18 + 4 = 22`.

⇒ `22` (giữ `0–1`, `2–3`, `2–4`; bỏ `0–2`).

## 5. Cách 1 — Brute Force: thử mọi tập cạnh

In [4]:
def max_weight_brute(edges, k):
    n = len(edges) + 1
    best = 0
    for mask in range(1 << len(edges)):
        deg = [0] * n; s = 0
        for i, (u, v, w) in enumerate(edges):
            if mask >> i & 1:
                deg[u] += 1; deg[v] += 1; s += w
        if max(deg) <= k:
            best = max(best, s)
    return best

for e, k, exp in examples:
    print(max_weight_brute(e, k), '| kỳ vọng', exp)

22 | kỳ vọng 22
65 | kỳ vọng 65


**Độ phức tạp:** `O(2^{n−1} · n)` — chỉ cho `n ≤ 15`.

## 6. Ý tưởng thuật toán / bước đột phá

Với đỉnh `u` có các con `v` (cạnh trọng số `w`):

- `base = Σ f[v][0]`, `gain_v = w + f[v][1] − f[v][0]`;
- `f[u][0] = base + (tổng tối đa k gain dương lớn nhất)`;
- `f[u][1] = base + (tổng tối đa k − 1 gain dương lớn nhất)`.

Đáp án `f[gốc][0]`. Sắp gain mỗi đỉnh: tổng `O(n log n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP đệ quy có nhớ (gọn, cần nâng giới hạn và stack lớn)

In [5]:
import sys, threading

def max_weight_recursive(edges, k):
    n = len(edges) + 1
    g = [[] for _ in range(n)]
    for u, v, w in edges:
        g[u].append((v, w)); g[v].append((u, w))
    res = {}
    def dfs(u, p):
        base, gains = 0, []
        for v, w in g[u]:
            if v != p:
                a, b = dfs(v, u)
                base += a; gains.append(w + b - a)
        gains.sort(reverse=True)
        return base + top_positive(gains, k), base + top_positive(gains, k - 1)
    def run():
        res['ans'] = dfs(0, -1)[0]
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(max(old, 3 * n + 100))
    threading.stack_size(256 * 1024 * 1024)
    th = threading.Thread(target=run); th.start(); th.join()
    threading.stack_size(0); sys.setrecursionlimit(old)
    return res['ans']

for e, k, exp in examples:
    print(max_weight_recursive(e, k), '| kỳ vọng', exp)

22 | kỳ vọng 22
65 | kỳ vọng 65


### 7.2. Cách 3 — DP lặp theo thứ tự BFS ngược (lời giải chính)

In [6]:
def max_weight_dp(edges, k):
    n = len(edges) + 1
    parent, pw, order = bfs_order(n, edges)
    f0 = [0] * n; f1 = [0] * n
    gains = [[] for _ in range(n)]
    base = [0] * n
    for u in reversed(order):
        gs = gains[u]
        gs.sort(reverse=True)
        s0 = s1 = base[u]
        for i, gval in enumerate(gs):
            if gval <= 0:
                break
            if i < k:
                s0 += gval
            if i < k - 1:
                s1 += gval
        f0[u], f1[u] = s0, s1
        p = parent[u]
        if p != -1:
            base[p] += f0[u]
            gains[p].append(pw[u] + f1[u] - f0[u])
    return f0[0]

for e, k, exp in examples:
    print(max_weight_dp(e, k), '| kỳ vọng', exp)

22 | kỳ vọng 22
65 | kỳ vọng 65


### 7.3. Cách 4 — Như cách 3 nhưng chọn top-k bằng `heapq.nlargest`

Khi bậc lớn mà `k` nhỏ, `nlargest(k, gains)` là `O(d log k)` thay cho sắp `O(d log d)`.

In [7]:
import heapq

def max_weight_nlargest(edges, k):
    n = len(edges) + 1
    parent, pw, order = bfs_order(n, edges)
    f0 = [0] * n; f1 = [0] * n
    gains = [[] for _ in range(n)]; base = [0] * n
    for u in reversed(order):
        top = [x for x in heapq.nlargest(k, gains[u]) if x > 0]
        f0[u] = base[u] + sum(top)
        f1[u] = base[u] + sum(top[:k - 1])
        p = parent[u]
        if p != -1:
            base[p] += f0[u]; gains[p].append(pw[u] + f1[u] - f0[u])
    return f0[0]

for e, k, exp in examples:
    print(max_weight_nlargest(e, k), '| kỳ vọng', exp)

22 | kỳ vọng 22
65 | kỳ vọng 65


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1.

In [8]:
def trace(edges, k):
    n = len(edges) + 1
    parent, pw, order = bfs_order(n, edges)
    f0 = [0] * n; f1 = [0] * n; gains = [[] for _ in range(n)]; base = [0] * n
    for u in reversed(order):
        gs = sorted(gains[u], reverse=True)
        pos = [x for x in gs if x > 0]
        f0[u] = base[u] + sum(pos[:k]); f1[u] = base[u] + sum(pos[:k - 1])
        print(f"đỉnh {u}: base = {base[u]}, gain con = {gs} -> f0 = {f0[u]}, f1 = {f1[u]}")
        p = parent[u]
        if p != -1:
            base[p] += f0[u]; gains[p].append(pw[u] + f1[u] - f0[u])
    print('đáp án:', f0[0])

trace(*examples[0][:2])

đỉnh 4: base = 0, gain con = [] -> f0 = 0, f1 = 0
đỉnh 3: base = 0, gain con = [] -> f0 = 0, f1 = 0
đỉnh 2: base = 0, gain con = [12, 6] -> f0 = 18, f1 = 12
đỉnh 1: base = 0, gain con = [] -> f0 = 0, f1 = 0
đỉnh 0: base = 18, gain con = [4, -4] -> f0 = 22, f1 = 22
đáp án: 22


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def maximizeSumOfWeights(self, edges: List[List[int]], k: int) -> int:
        n = len(edges) + 1
        g = [[] for _ in range(n)]
        for u, v, w in edges:
            g[u].append((v, w))
            g[v].append((u, w))
        parent = [-1] * n
        pw = [0] * n
        order = [0]
        seen = [False] * n
        seen[0] = True
        for u in order:
            for v, w in g[u]:
                if not seen[v]:
                    seen[v] = True
                    parent[v] = u
                    pw[v] = w
                    order.append(v)
        f0 = [0] * n
        f1 = [0] * n
        base = [0] * n
        gains = [[] for _ in range(n)]
        for u in reversed(order):
            gs = gains[u]
            gs.sort(reverse=True)
            s0 = s1 = base[u]
            for i, x in enumerate(gs):
                if x <= 0 or i >= k:
                    break
                s0 += x
                if i < k - 1:
                    s1 += x
            f0[u] = s0
            f1[u] = s1
            p = parent[u]
            if p != -1:
                base[p] += s0
                gains[p].append(pw[u] + s1 - s0)
        return f0[0]

for e, k, exp in examples:
    got = Solution().maximizeSumOfWeights(e, k)
    print(got, 'OK' if got == exp else 'SAI')

22 OK
65 OK


## 10. Vì sao thuật toán đúng

1. **Tách cây con:** khi đã quyết định cạnh `u – cha` giữ hay bỏ, các cây con của `u` độc lập với phần còn lại ngoại trừ ràng buộc bậc tại `u`.
2. **Mỗi con:** hoặc bỏ cạnh `u–v` (đóng góp tốt nhất `f[v][0]`), hoặc giữ (đóng góp `w + f[v][1]`, vì `v` đã dùng một cạnh). Chênh lệch là `gain_v`.
3. **Ràng buộc bậc tại `u`:** số cạnh con giữ ≤ `k` (hoặc `k − 1` nếu cạnh lên cha giữ). Bài con "chọn tối đa `t` phần tử để tổng lợi ích lớn nhất" ⇒ lấy các `gain` dương lớn nhất.
4. Quy nạp từ lá lên gốc ⇒ `f[gốc][0]` tối ưu.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi tập cạnh | `O(2^n · n)` | `O(n)` |
| 2. Đệ quy | `O(n log n)` | `O(n)` + stack |
| 3. DP lặp + sắp | `O(n log n)` | `O(n)` |
| 4. DP + `nlargest` | `O(Σ d log k)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'recursive': max_weight_recursive, 'dp': max_weight_dp, 'nlargest': max_weight_nlargest,
         'submit': lambda e, k: Solution().maximizeSumOfWeights(e, k)}
edge = [([[0, 1, 7]], 1, 7), ([[0, 1, 1], [0, 2, 2], [0, 3, 3]], 1, 3), ([[0, 1, 1], [0, 2, 2], [0, 3, 3]], 2, 5),
        ([[0, 1, 5], [1, 2, 5], [2, 3, 5]], 1, 10)]
for e, k, exp in examples + edge:
    assert max_weight_brute(e, k) == exp, (e, k, max_weight_brute(e, k))
    for name, f in impls.items():
        assert f(e, k) == exp, (name, e, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random

def random_tree(n, wmax, rng):
    perm = list(range(n)); rng.shuffle(perm)
    return [[perm[v], perm[rng.randrange(v)], rng.randint(1, wmax)] for v in range(1, n)]

rng = random.Random(3367)
cnt = 0
for _ in range(700):
    n = rng.randint(2, 11)
    e = random_tree(n, rng.choice([3, 20]), rng)
    k = rng.randint(1, n - 1)
    exp = max_weight_brute(e, k)
    for name, f in impls.items():
        assert f(e, k) == exp, (name, e, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'cây ngẫu nhiên.')

Khớp vét cạn trên 700 cây ngẫu nhiên.


### 12.3. Ca lớn `n = 10^5`

In [12]:
import time
n = 10 ** 5
cases = {
    'ngẫu nhiên, k = 2': (random_tree(n, 10 ** 6, rng), 2),
    'sao, k = 3': ([[0, i, rng.randint(1, 10 ** 6)] for i in range(1, n)], 3),
    'đường, k = 1': ([[i, i + 1, rng.randint(1, 10 ** 6)] for i in range(n - 1)], 1),
}
for name, (e, k) in cases.items():
    t = time.time(); a = Solution().maximizeSumOfWeights(e, k); t1 = time.time() - t
    t = time.time(); b = max_weight_nlargest(e, k); t2 = time.time() - t
    assert a == b
    print(f"{name:20s}: {a:14d} | sắp {t1:.2f}s | nlargest {t2:.2f}s")

ngẫu nhiên, k = 2   :    38184425562 | sắp 0.47s | nlargest 0.53s


sao, k = 3          :        2999940 | sắp 0.22s | nlargest 0.18s


đường, k = 1        :    29343977493 | sắp 0.26s | nlargest 0.19s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: duyệt cạnh từ nặng tới nhẹ, giữ nếu hai đầu còn "chỗ"

In [13]:
def wrong_greedy(edges, k):
    n = len(edges) + 1; deg = [0] * n; s = 0
    for u, v, w in sorted(edges, key=lambda e: -e[2]):
        if deg[u] < k and deg[v] < k:
            deg[u] += 1; deg[v] += 1; s += w
    return s

rng = random.Random(1)
ce = next((e, k) for e, k in ((random_tree(rng.randint(3, 8), 10, rng), rng.randint(1, 2)) for _ in range(20000))
          if wrong_greedy(e, k) != max_weight_brute(e, k))
print('phản ví dụ:', ce, '| sai:', wrong_greedy(*ce), '| đúng:', max_weight_brute(*ce))

phản ví dụ: ([[1, 3, 8], [0, 3, 10], [2, 0, 6]], 1) | sai: 10 | đúng: 14


**Vì sao sai:** giữ một cạnh nặng có thể "chiếm chỗ" của hai cạnh nhẹ hơn nhưng tổng lớn hơn (bài toán b-matching trên cây cần DP).

### Lỗi 2 — Cộng cả lợi ích âm (luôn lấy đủ `k` con)

In [14]:
def wrong_take_negative(edges, k):
    n = len(edges) + 1
    parent, pw, order = bfs_order(n, edges)
    f0 = [0] * n; f1 = [0] * n; gains = [[] for _ in range(n)]; base = [0] * n
    for u in reversed(order):
        gs = sorted(gains[u], reverse=True)
        f0[u] = base[u] + sum(gs[:k]); f1[u] = base[u] + sum(gs[:k - 1])      # SAI: không lọc gain <= 0
        p = parent[u]
        if p != -1:
            base[p] += f0[u]; gains[p].append(pw[u] + f1[u] - f0[u])
    return f0[0]

print('sai :', wrong_take_negative(*examples[0][:2]), '| đúng:', Solution().maximizeSumOfWeights(*examples[0][:2]))

sai : 18 | đúng: 22


**Vì sao sai:** giữ cạnh `0–2` có lợi ích `−4` (vì buộc đỉnh 2 bỏ một cạnh nặng hơn); "tối đa `k`" cho phép **không** giữ.

### Lỗi 3 — `f[u][1]` vẫn cho dùng `k` cạnh con

In [15]:
def wrong_k_for_f1(edges, k):
    n = len(edges) + 1
    parent, pw, order = bfs_order(n, edges)
    f0 = [0] * n; f1 = [0] * n; gains = [[] for _ in range(n)]; base = [0] * n
    for u in reversed(order):
        pos = [x for x in sorted(gains[u], reverse=True) if x > 0]
        f0[u] = f1[u] = base[u] + sum(pos[:k])                               # SAI
        p = parent[u]
        if p != -1:
            base[p] += f0[u]; gains[p].append(pw[u] + f1[u] - f0[u])
    return f0[0]

print('sai :', wrong_k_for_f1(*examples[0][:2]), '| đúng:', Solution().maximizeSumOfWeights(*examples[0][:2]))

sai : 24 | đúng: 22


**Vì sao sai:** khi giữ cạnh lên cha, đỉnh đã dùng một "chỗ" bậc ⇒ chỉ còn `k − 1` cạnh con.

### Lỗi 4 — DFS đệ quy với giới hạn mặc định

In [16]:
def naive_recursive(edges, k):
    n = len(edges) + 1
    g = [[] for _ in range(n)]
    for u, v, w in edges:
        g[u].append((v, w)); g[v].append((u, w))
    def dfs(u, p):
        return sum(dfs(v, u) for v, _ in g[u] if v != p) + 1
    return dfs(0, -1)

try:
    naive_recursive(cases['đường, k = 1'][0], 1)
except RecursionError as ex:
    print('RecursionError:', ex)

RecursionError: maximum recursion depth exceeded


**Vì sao sai:** cây dạng đường `10^5` đỉnh vượt giới hạn đệ quy mặc định.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn cạnh trên cây với ràng buộc bậc" ⇒ DP hai trạng thái theo cạnh lên cha.
- "Chọn tối đa `k` con có lợi" ⇒ sắp lợi ích `gain = (giữ) − (bỏ)`, lấy các giá trị dương lớn nhất.
- `k = 1` ⇒ ghép cặp cực đại có trọng số trên cây.
- Duyệt BFS ngược thay đệ quy.

## 15. Biến thể của bài

1. `k = 1`: **maximum weight matching** trên cây.
2. Mỗi đỉnh có giới hạn bậc riêng `k_v` ⇒ thay `k` bằng `k_v`.
3. Đếm số cách đạt tối ưu ⇒ DP thêm số cách (khó khi có hòa lợi ích).
4. Bài liên quan: LC 2378 (Choose Edges to Maximize Score in a Tree) — chính là `k = 1`.

In [17]:
# Biến thể 2: giới hạn bậc riêng cho từng đỉnh
def max_weight_per_vertex(edges, caps):
    n = len(edges) + 1
    parent, pw, order = bfs_order(n, edges)
    f0 = [0] * n; f1 = [0] * n; gains = [[] for _ in range(n)]; base = [0] * n
    for u in reversed(order):
        pos = [x for x in sorted(gains[u], reverse=True) if x > 0]
        f0[u] = base[u] + sum(pos[:caps[u]])
        f1[u] = base[u] + sum(pos[:caps[u] - 1]) if caps[u] >= 1 else -10 ** 18
        p = parent[u]
        if p != -1:
            base[p] += f0[u]; gains[p].append(pw[u] + f1[u] - f0[u])
    return f0[0]

def brute_caps(edges, caps):
    n = len(edges) + 1; best = 0
    for mask in range(1 << len(edges)):
        deg = [0] * n; s = 0
        for i, (u, v, w) in enumerate(edges):
            if mask >> i & 1:
                deg[u] += 1; deg[v] += 1; s += w
        if all(deg[i] <= caps[i] for i in range(n)):
            best = max(best, s)
    return best

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(2, 9); e = random_tree(n, 10, rng); caps = [rng.randint(0, 3) for _ in range(n)]
    assert max_weight_per_vertex(e, caps) == brute_caps(e, caps), (e, caps)
print('Biến thể giới hạn bậc riêng (kể cả 0) khớp vét cạn.')

Biến thể giới hạn bậc riêng (kể cả 0) khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi tập cạnh | mũ | Oracle |
| 2 | DP đệ quy | `O(n log n)` | Cần stack lớn |
| 3 | DP BFS ngược | `O(n log n)` | **Nộp bài** |
| 4 | DP + `nlargest` | `O(Σ d log k)` | Khi `k` nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** sao 4 cạnh trọng số `1,2,3,4`, `k = 2`.

**Bài 2 (tính tay):** đường `0–1–2–3` trọng số `3, 1, 3`, `k = 1`.

**Bài 3 (điền TODO):** trả về **tập cạnh** được giữ.

**Bài 4 (tự viết lại):** LC 2378 — `k = 1`, trọng số có thể âm.

**Bài 5 (mini challenge):** biến thể: mỗi đỉnh phải có bậc **chính xác** `≤ k` và đồ thị còn lại phải **liên thông**? (thử tìm phản ví dụ cho việc dùng lại DP này).

In [18]:
def kept_edges_todo(edges, k):
    # TODO
    raise NotImplementedError

def max_matching_score_todo(parent_edges):
    # TODO
    raise NotImplementedError

def connected_variant_todo(edges, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tâm bậc 4 > 2. (2) Giữ hai cạnh nặng nhất. (3) `7`.

**Bài 2.** (1) `k = 1` ⇒ ghép cặp. (2) `{0–1, 2–3}` = 6. (3) `6`.

**Bài 3.** (1) Lưu các con được chọn ở mỗi đỉnh theo trạng thái. (2) Truy vết từ gốc trạng thái 0. (3) Kiểm tra bậc và tổng.

**Bài 4.** (1) `f1[u] = base` (không chọn thêm con), `f0[u] = base + max(0, max gain)`. (2) Trọng số âm ⇒ gain âm ⇒ không chọn. (3) Dạng `parent[i]`, `weight[i]`.

**Bài 5.** (1) Nếu phải liên thông thì không được bỏ cạnh nào (cây!) ⇒ khả thi ⇔ bậc lớn nhất ≤ k. (2) DP trên không cần. (3) Kiểm tra trực tiếp.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2
assert Solution().maximizeSumOfWeights([[0, 1, 1], [0, 2, 2], [0, 3, 3], [0, 4, 4]], 2) == 7
assert Solution().maximizeSumOfWeights([[0, 1, 3], [1, 2, 1], [2, 3, 3]], 1) == 6
print('Bài 1: 7 | Bài 2: 6')

# Bài 3 — truy vết
def kept_edges(edges, k):
    n = len(edges) + 1
    parent, pw, order = bfs_order(n, edges)
    f0 = [0] * n; f1 = [0] * n; base = [0] * n
    kids = [[] for _ in range(n)]
    for u in reversed(order):
        pos = sorted(((pw[v] + f1[v] - f0[v], v) for v in kids[u]), reverse=True)
        pos = [(gv, v) for gv, v in pos if gv > 0]
        f0[u] = base[u] + sum(gv for gv, _ in pos[:k]); f1[u] = base[u] + sum(gv for gv, _ in pos[:k - 1])
        p = parent[u]
        if p != -1:
            base[p] += f0[u]; kids[p].append(u)
    kept = []
    stack = [(0, 0)]
    while stack:
        u, state = stack.pop()
        pos = sorted(((pw[v] + f1[v] - f0[v], v) for v in kids[u]), reverse=True)
        chosen = {v for gv, v in pos[:k - state] if gv > 0}
        for v in kids[u]:
            if v in chosen:
                kept.append((parent[v], v, pw[v])); stack.append((v, 1))
            else:
                stack.append((v, 0))
    return f0[0], kept

rng = random.Random(3)
for _ in range(300):
    n = rng.randint(2, 30); e = random_tree(n, 20, rng); k = rng.randint(1, 3)
    val, kept = kept_edges(e, k)
    deg = [0] * n
    for u, v, w in kept:
        deg[u] += 1; deg[v] += 1
    assert val == Solution().maximizeSumOfWeights(e, k) == sum(w for _, _, w in kept) and max(deg) <= k
print('Bài 3: tập cạnh hợp lệ; ví dụ 1 ->', kept_edges(*examples[0][:2]))

Bài 1: 7 | Bài 2: 6


Bài 3: tập cạnh hợp lệ; ví dụ 1 -> (22, [(0, 1, 4), (2, 4, 6), (2, 3, 12)])


In [20]:
# Bài 4 (LC 2378), Bài 5
def max_matching_score(edges_pw):
    # edges_pw[i] = [parent, weight], edges_pw[0] = [-1, -1]
    n = len(edges_pw)
    edges = [[p, i, w] for i, (p, w) in enumerate(edges_pw) if i > 0]
    parent, pw, order = bfs_order(n, edges)
    f0 = [0] * n; f1 = [0] * n; gains = [[] for _ in range(n)]; base = [0] * n
    for u in reversed(order):
        best = max([x for x in gains[u] if x > 0], default=0)
        f0[u] = base[u] + best; f1[u] = base[u]
        p = parent[u]
        if p != -1:
            base[p] += f0[u]; gains[p].append(pw[u] + f1[u] - f0[u])
    return f0[0]

assert max_matching_score([[-1, -1], [0, 5], [0, 10], [2, 6], [2, 4]]) == 11          # ví dụ LC 2378
assert max_matching_score([[-1, -1], [0, 5], [0, -6], [0, 7]]) == 7

def connected_variant(edges, k):
    n = len(edges) + 1; deg = [0] * n
    for u, v, _ in edges:
        deg[u] += 1; deg[v] += 1
    return sum(w for _, _, w in edges) if max(deg) <= k else -1

print('Bài 4: LC 2378 ví dụ đúng | Bài 5:', connected_variant(*examples[0][:2]), connected_variant(*examples[1][:2]))

Bài 4: LC 2378 ví dụ đúng | Bài 5: -1 65


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
for u in BFS ngược:
    gains[u] sắp giảm; f0[u] = base[u] + Σ k gain dương lớn nhất
                        f1[u] = base[u] + Σ (k-1) gain dương lớn nhất
    cha p: base[p] += f0[u]; gains[p].append(w(u,p) + f1[u] - f0[u])
return f0[gốc]
```

**Checklist:**

- [ ] Hai trạng thái: cạnh lên cha giữ/bỏ?
- [ ] Chỉ lấy gain **dương**?
- [ ] `f1` dùng `k − 1` con?
- [ ] Không đệ quy sâu?
- [ ] Tổng tới `10^{11}` (C++/Java cần `long long`)?